# Verovio vs Partitura parsing

Use this notebook to compare CAMAT's default `partitura` common-notation parser with the experimental `verovio` MEI parser. The setup mirrors the first parsing cell in `testing_annot_stats.ipynb`, but plotting and previews are off by default so the output stays focused on parser parity.

In [1]:
# Imports and shared parser options
from __future__ import annotations

import math
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, clear_output, display

from camat.parser_registry import parse_files

# ---------------------------------------------------------------------
# Scope: common-notation MEI parser comparison only.
# The Verovio backend currently supports common-notation MEI sources.
# Mensural notation belongs in testing_parser_mensural.ipynb.
# ---------------------------------------------------------------------

BACKENDS = ("partitura", "verovio")
PRINT_PARSED_SUMMARY = False

FILTER_ZERO_DURATION = True
ADJUST_FRACTIONAL_DURATION = True
PARSE_ENHARMONIC = True
DEDUPE_WEAKER_TEXT_EVENTS = False

PLOTTING_BACKEND = "none"
SHOW_MEASURE_LINES = False
MEASURE_LINE_COLOR = "black"
SHOW_HOVER = True
HOVER_FIELDS = [
    "measure", "local_onset", "global_onset", "duration",
    "pitch", "pitch_enharmonic", "midi", "voice", "xml_id",
]

DISPLAY_PREVIEW_DF_PITCH = False
DISPLAY_PREVIEW_DF_EVENTS = False
PREVIEW_ROWS = 20
CLEANUP_REMOTE = True
RETURN_PLOTS = False
PLOT_SIZE_X = 900
PLOT_SIZE_Y = 600
ZOOM_DRAG_DIM = "both"
ZOOM_WHEEL_DIM = "width"
SHOW_PROGRESS = True
PROGRESS_DESC = None

QUIET_NATIVE_WARNINGS = True
PARALLEL_N_JOBS = 1
USE_REMOTE_CACHE = True
REMOTE_CACHE_DIR = None

COLLAPSE_TIED_PITCH_EVENTS = False
ALIGN_ACCIDENT_SCHEMA = True
COLORIZE_VOICES = True
PALETTE = "Category20"
INCLUDE_XML_IDS = True

TRY_VEROVIO_MEI_CONVERSION = True
PLOT_PARSED_BARLINES_WITH_VOICE_COLORING = True
ALLOW_MUSIC21_FALLBACK = False

# Start from the same source block as testing_annot_stats.ipynb.
# Uncomment the Mozart / Beethoven examples when you want broader parity checks.
FILE_SOURCES = [
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_3.0/Music/Complete_examples/Mozart_Fuge_G_minor.mei",
    # "https://raw.githubusercontent.com/trompamusic-encodings/Beethoven_Op31_No3_HenleUrtext/refs/heads/master/Beethoven_Op31_No3_3-HenleUrtext.mei",
    "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Bach-JS_BrandenburgConcert_No4_I_BWV1049.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Beethoven_StringQuartet_Op18_No1.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Brahms_StringQuartet_Op51_No1.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Chopin_Etude_Op10_No9.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Chopin_Mazurka_Op6_No1.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Haydn_StringQuartet_Op1_No1.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Joplin_Maple_leaf_Rag.mei",
    # "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Schumann_Song_Op48_No1.mei",
]

COMMON_PARSE_KWARGS = dict(
    quiet_native_warnings=QUIET_NATIVE_WARNINGS,
    dedupe_weaker_text_events=DEDUPE_WEAKER_TEXT_EVENTS,
    print_parsed_summary=PRINT_PARSED_SUMMARY,
    filter_zero_duration=FILTER_ZERO_DURATION,
    adjust_fractional_duration=ADJUST_FRACTIONAL_DURATION,
    parse_enharmonic=PARSE_ENHARMONIC,
    backend=PLOTTING_BACKEND,
    show_measure_lines=SHOW_MEASURE_LINES,
    measure_line_color=MEASURE_LINE_COLOR,
    show_hover=SHOW_HOVER,
    hover_fields=HOVER_FIELDS,
    display_preview_df_pitch=DISPLAY_PREVIEW_DF_PITCH,
    display_preview_df_events=DISPLAY_PREVIEW_DF_EVENTS,
    preview_rows=PREVIEW_ROWS,
    cleanup_remote=CLEANUP_REMOTE,
    return_plots=RETURN_PLOTS,
    plot_width=PLOT_SIZE_X,
    plot_height=PLOT_SIZE_Y,
    zoom_drag_dim=ZOOM_DRAG_DIM,
    zoom_wheel_dim=ZOOM_WHEEL_DIM,
    show_progress=SHOW_PROGRESS,
    progress_desc=PROGRESS_DESC,
    collapse_tied_pitch_events=COLLAPSE_TIED_PITCH_EVENTS,
    align_accident_schema=ALIGN_ACCIDENT_SCHEMA,
    colorize_voices=COLORIZE_VOICES,
    palette=PALETTE,
    include_xml_ids=INCLUDE_XML_IDS,
    try_verovio_mei_conversion=TRY_VEROVIO_MEI_CONVERSION,
    plot_parsed_barlines_with_voice_coloring=PLOT_PARSED_BARLINES_WITH_VOICE_COLORING,
    allow_music21_fallback=ALLOW_MUSIC21_FALLBACK,
    use_remote_cache=USE_REMOTE_CACHE,
    remote_cache_dir=REMOTE_CACHE_DIR,
    n_jobs=PARALLEL_N_JOBS,
    normalize_mensural_durations=False,
    inject_missing_meter_signature=False,
    prefer_verovio_for_mensural=False,
    use_verovio_mensural_timing=False,
)

display(FILE_SOURCES)


['https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.1/Music/Complete_examples/Bach-JS_BrandenburgConcert_No4_I_BWV1049.mei']

In [2]:
# Parse every source with Partitura and Verovio.

def parse_with_backend(backend: str):
    kwargs = dict(COMMON_PARSE_KWARGS)
    kwargs["parsing_backend"] = backend
    print(f"Parsing with {backend}...")
    return parse_files(FILE_SOURCES, **kwargs)


parsed_by_backend = {backend: parse_with_backend(backend) for backend in BACKENDS}
results_by_backend = {backend: parsed_by_backend[backend][0] for backend in BACKENDS}
dfs_by_backend = {backend: parsed_by_backend[backend][1] for backend in BACKENDS}
last_df_by_backend = {backend: parsed_by_backend[backend][2] for backend in BACKENDS}

source_names = [entry["name"] for entry in results_by_backend["partitura"]]
display(pd.DataFrame({"source_index": range(len(source_names)), "name": source_names, "source": FILE_SOURCES}))


Parsing with partitura...
Processing (partitura): Bach-JS_BrandenburgConcert_No4_I_BWV1049.mei -> 00_bach_js_brandenburgconcert_no4_i_bwv1049
Extracted non-barline MEI events: 1857 event(s), types=['direction', 'fermata', 'measure', 'mrest', 'slur', 'tempo', 'tie', 'trill'].
Extracted rest timing events: 3607 event(s).
Parsing with verovio...
Processing (verovio): Bach-JS_BrandenburgConcert_No4_I_BWV1049.mei -> 00_bach_js_brandenburgconcert_no4_i_bwv1049
Extracted non-barline MEI events: 1857 event(s), types=['direction', 'fermata', 'measure', 'mrest', 'slur', 'tempo', 'tie', 'trill'].


,source_index,name,source
0,0,00_bach_js_brandenburgconcert_no4_i_bwv1049,https://raw.githubusercontent.com/music-encodi...


In [3]:
# Comparison helpers for df_pitch.

PITCH_COMPARE_COLUMNS = [
    "MIDI",
    "Global Onset",
    "Local Onset",
    "Duration",
    "Pitch",
    "Pitch Enharmonic",
    "Voice",
    "Measure",
]
NUMERIC_COMPARE_COLUMNS = ["MIDI", "Global Onset", "Local Onset", "Duration", "Measure"]
# Core parity intentionally ignores backend-specific voice labels.
# Use `voices` or `strict` when you explicitly want to inspect label drift.
CORE_COMPARE_COLUMNS = ["MIDI", "Global Onset", "Duration", "Pitch"]
TIMING_COMPARE_COLUMNS = ["Global Onset", "Duration"]
LOCATION_COMPARE_COLUMNS = ["Measure", "Local Onset"]
SPELLING_COMPARE_COLUMNS = ["Pitch Enharmonic"]
VOICE_COMPARE_COLUMNS = ["Voice"]
COMPARE_COLUMN_PRESETS = {
    "core": CORE_COMPARE_COLUMNS,
    "timing": TIMING_COMPARE_COLUMNS,
    "strict": PITCH_COMPARE_COLUMNS,
    "location": LOCATION_COMPARE_COLUMNS,
    "spelling": SPELLING_COMPARE_COLUMNS,
    "voices": VOICE_COMPARE_COLUMNS,
}


def get_result(backend: str, source_index: int = 0) -> dict:
    return results_by_backend[backend][source_index]


def get_pitch_df(backend: str, source_index: int = 0) -> pd.DataFrame:
    return get_result(backend, source_index)["df_pitch"].copy()


def get_events_df(backend: str, source_index: int = 0) -> pd.DataFrame:
    return get_result(backend, source_index)["df_events"].copy()


def _ensure_xml_id(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "xml_id" not in out.columns:
        out["xml_id"] = pd.NA
    out["xml_id"] = out["xml_id"].astype("string")
    return out


def compare_pitch_dfs(
    source_index: int = 0,
    tolerance: float = 1e-6,
    compare_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    active_compare_columns = list(compare_columns or CORE_COMPARE_COLUMNS)
    part = _ensure_xml_id(get_pitch_df("partitura", source_index))
    vrv = _ensure_xml_id(get_pitch_df("verovio", source_index))
    keep_cols = ["xml_id"] + [c for c in PITCH_COMPARE_COLUMNS if c in part.columns or c in vrv.columns]
    for col in keep_cols:
        if col not in part.columns:
            part[col] = pd.NA
        if col not in vrv.columns:
            vrv[col] = pd.NA
    merged = part[keep_cols].merge(
        vrv[keep_cols],
        on="xml_id",
        how="outer",
        suffixes=("_partitura", "_verovio"),
        indicator=True,
    )

    statuses = []
    mismatch_columns = []
    for _, row in merged.iterrows():
        if row["_merge"] == "left_only":
            statuses.append("missing_in_verovio")
            mismatch_columns.append("xml_id")
            continue
        if row["_merge"] == "right_only":
            statuses.append("extra_in_verovio")
            mismatch_columns.append("xml_id")
            continue

        diffs = []
        for col in active_compare_columns:
            left_col = f"{col}_partitura"
            right_col = f"{col}_verovio"
            if left_col not in merged.columns or right_col not in merged.columns:
                continue
            left = row[left_col]
            right = row[right_col]
            if col in NUMERIC_COMPARE_COLUMNS:
                left_num = pd.to_numeric(pd.Series([left]), errors="coerce").iloc[0]
                right_num = pd.to_numeric(pd.Series([right]), errors="coerce").iloc[0]
                if pd.isna(left_num) and pd.isna(right_num):
                    continue
                if pd.isna(left_num) != pd.isna(right_num) or abs(float(left_num) - float(right_num)) > tolerance:
                    diffs.append(col)
            else:
                if pd.isna(left) and pd.isna(right):
                    continue
                if str(left) != str(right):
                    diffs.append(col)
        statuses.append("match" if not diffs else "value_mismatch")
        mismatch_columns.append(", ".join(diffs))

    merged["status"] = statuses
    merged["mismatch_columns"] = mismatch_columns
    status_order = {
        "missing_in_verovio": 0,
        "extra_in_verovio": 1,
        "value_mismatch": 2,
        "match": 3,
    }
    merged["_status_order"] = merged["status"].map(status_order).fillna(9)
    sort_cols = ["_status_order"]
    if "Global Onset_partitura" in merged.columns:
        sort_cols.append("Global Onset_partitura")
    if "Global Onset_verovio" in merged.columns:
        sort_cols.append("Global Onset_verovio")
    merged = merged.sort_values(sort_cols, na_position="last").drop(columns=["_status_order"])
    return merged.reset_index(drop=True)


def parity_summary(
    source_index: int = 0,
    tolerance: float = 1e-6,
    compare_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    cmp = compare_pitch_dfs(source_index, tolerance=tolerance, compare_columns=compare_columns)
    part = get_pitch_df("partitura", source_index)
    vrv = get_pitch_df("verovio", source_index)
    counts = cmp["status"].value_counts().to_dict()
    rows = [{
        "source": source_names[source_index],
        "partitura_rows": len(part),
        "verovio_rows": len(vrv),
        "partitura_xml_ids": part.get("xml_id", pd.Series(dtype=object)).dropna().nunique(),
        "verovio_xml_ids": vrv.get("xml_id", pd.Series(dtype=object)).dropna().nunique(),
        "matches": counts.get("match", 0),
        "value_mismatches": counts.get("value_mismatch", 0),
        "missing_in_verovio": counts.get("missing_in_verovio", 0),
        "extra_in_verovio": counts.get("extra_in_verovio", 0),
    }]
    return pd.DataFrame(rows)


def parity_summary_all_sources(compare_mode: str = "core", tolerance: float = 1e-6) -> pd.DataFrame:
    compare_columns = COMPARE_COLUMN_PRESETS[compare_mode]
    frames = []
    for i in range(len(source_names)):
        summary = parity_summary(i, tolerance=tolerance, compare_columns=compare_columns)
        summary.insert(1, "comparison", compare_mode)
        frames.append(summary)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def filtered_mismatches(
    source_index: int = 0,
    compare_mode: str = "core",
    status: str = "value_mismatch",
    tolerance: float = 1e-6,
    xml_id_filter: str = "",
) -> pd.DataFrame:
    cmp = compare_pitch_dfs(
        source_index,
        tolerance=tolerance,
        compare_columns=COMPARE_COLUMN_PRESETS[compare_mode],
    )
    if status != "all":
        cmp = cmp[cmp["status"] == status]
    if xml_id_filter.strip():
        needle = xml_id_filter.strip()
        cmp = cmp[cmp["xml_id"].astype(str).str.contains(needle, case=False, na=False)]
    return cmp.reset_index(drop=True)


display(pd.concat([
    parity_summary_all_sources("core"),
    parity_summary_all_sources("strict"),
], ignore_index=True))


,source,comparison,partitura_rows,verovio_rows,partitura_xml_ids,verovio_xml_ids,matches,value_mismatches,missing_in_verovio,extra_in_verovio
0,00_bach_js_brandenburgconcert_no4_i_bwv1049,core,10763,10763,10763,10763,10763,0,0,0
1,00_bach_js_brandenburgconcert_no4_i_bwv1049,strict,10763,10763,10763,10763,8428,2335,0,0


In [4]:
# Side-by-side df_pitch display helpers.

SIDE_BY_SIDE_COLUMNS = [
    "Measure", "Local Onset", "Global Onset", "Duration",
    "Pitch", "Pitch Enharmonic", "MIDI", "Voice", "xml_id",
]


def display_scrollable_df(df: pd.DataFrame, max_height: int = 360) -> None:
    html = f"""
    <div style="max-height: {int(max_height)}px; overflow: auto; border: 1px solid #ddd; padding: 6px;">
      {df.to_html(index=False, escape=True)}
    </div>
    """
    display(HTML(html))


def display_side_by_side(part_df: pd.DataFrame, vrv_df: pd.DataFrame, n: int = 25, title: str = "") -> None:
    cols = [c for c in SIDE_BY_SIDE_COLUMNS if c in part_df.columns or c in vrv_df.columns]
    left = part_df.reindex(columns=cols).head(n)
    right = vrv_df.reindex(columns=cols).head(n)
    left_html = left.to_html(index=False, escape=True)
    right_html = right.to_html(index=False, escape=True)
    html = f"""
    <style>
      .camat-compare-wrap {{ display: flex; gap: 18px; align-items: flex-start; max-height: 520px; overflow-y: auto; border: 1px solid #ddd; padding: 6px; }}
      .camat-compare-pane {{ flex: 1 1 0; min-width: 0; overflow: auto; max-height: 500px; }}
      .camat-compare-pane h4 {{ margin: 0 0 6px 0; font-size: 14px; }}
      .camat-compare-pane table {{ font-size: 12px; }}
    </style>
    <h3>{title}</h3>
    <div class="camat-compare-wrap">
      <div class="camat-compare-pane"><h4>Partitura df_pitch</h4>{left_html}</div>
      <div class="camat-compare-pane"><h4>Verovio df_pitch</h4>{right_html}</div>
    </div>
    """
    display(HTML(html))


def show_pitch_window(source_index: int = 0, start: int = 0, n: int = 25) -> None:
    name = source_names[source_index]
    part = get_pitch_df("partitura", source_index).iloc[start:start + n]
    vrv = get_pitch_df("verovio", source_index).iloc[start:start + n]
    display_side_by_side(part, vrv, n=n, title=f"{name}: rows {start}..{start + n - 1}")


def _context_around_xml_id(df: pd.DataFrame, xml_id: str, context: int = 5) -> Tuple[pd.DataFrame, int]:
    if "xml_id" not in df.columns or not xml_id:
        return df.head(0), -1
    matches = df.index[df["xml_id"].astype(str) == str(xml_id)].tolist()
    if not matches:
        return df.head(0), -1
    pos = int(matches[0])
    start = max(0, pos - context)
    end = min(len(df), pos + context + 1)
    return df.iloc[start:end].copy(), pos


def display_mismatch_detail(
    source_index: int = 0,
    compare_mode: str = "core",
    mismatch_index: int = 0,
    tolerance: float = 1e-6,
    status: str = "value_mismatch",
    xml_id_filter: str = "",
    context: int = 5,
) -> Optional[str]:
    rows = filtered_mismatches(
        source_index=source_index,
        compare_mode=compare_mode,
        status=status,
        tolerance=tolerance,
        xml_id_filter=xml_id_filter,
    )
    if rows.empty:
        display(Markdown("No rows match the current mismatch filters."))
        return None
    idx = max(0, min(int(mismatch_index), len(rows) - 1))
    selected = rows.iloc[[idx]].copy()
    xml_id = str(selected.iloc[0]["xml_id"])
    display(Markdown(f"#### Selected mismatch {idx + 1}/{len(rows)}: `{xml_id}`"))
    display_scrollable_df(selected, max_height=220)

    part = get_pitch_df("partitura", source_index)
    vrv = get_pitch_df("verovio", source_index)
    part_context, part_pos = _context_around_xml_id(part, xml_id, context=context)
    vrv_context, vrv_pos = _context_around_xml_id(vrv, xml_id, context=context)
    display(Markdown(f"Partitura row index: `{part_pos}`; Verovio row index: `{vrv_pos}`"))
    display_side_by_side(
        part_context,
        vrv_context,
        n=(2 * context + 1),
        title=f"Context around {xml_id}",
    )
    return xml_id


show_pitch_window(source_index=0, start=0, n=25)


Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
1,0.00,0.00,0.50,G3,G3,55,Cello - Voice 1,d1e780
1,0.00,0.00,0.50,G3,G3,55,Double Bass - Voice 1,d1e839
1,0.00,0.00,0.50,G3,G3,55,w15rfroz - Voice 1,d1e974
1,0.00,0.00,0.50,B4,B4,71,Viola - Voice 1,d1e728
1,0.00,0.00,0.50,B4,B4,71,P9-I9 - Voice 1,d1e901
1,0.00,0.00,0.50,D5,D5,74,Violin II - Voice 1,d1e676
1,0.00,0.00,0.50,D5,D5,74,P9-I9 - Voice 1,d1e917
1,0.00,0.00,0.50,G5,G5,79,Solo Violin - Voice 1,d1e402
1,0.00,0.00,0.50,G5,G5,79,Violin I - Voice 1,d1e624
1,0.00,0.00,0.50,G5,G5,79,P9-I9 - Voice 1,d1e934


In [ ]:
# Interactive comparison interface.

try:
    import ipywidgets as widgets
except Exception as exc:
    widgets = None
    print(f"ipywidgets is unavailable: {exc}")


def render_comparison_view(
    source_index: int,
    compare_mode: str,
    status: str,
    mismatch_index: int,
    row_start: int,
    rows: int,
    tolerance: float,
    xml_id: str = "",
) -> None:
    clear_output(wait=True)
    name = source_names[source_index]
    part = get_pitch_df("partitura", source_index)
    vrv = get_pitch_df("verovio", source_index)
    compare_columns = COMPARE_COLUMN_PRESETS[compare_mode]
    cmp = filtered_mismatches(
        source_index=source_index,
        compare_mode=compare_mode,
        status=status,
        tolerance=tolerance,
        xml_id_filter=xml_id,
    )

    display(Markdown(f"### {name}"))
    display(Markdown(f"Comparison mode: `{compare_mode}` columns = `{', '.join(compare_columns)}`"))
    display(parity_summary(source_index, tolerance=tolerance, compare_columns=compare_columns))
    display(Markdown("#### Mismatch profile"))
    display_scrollable_df(cmp["mismatch_columns"].value_counts().rename_axis("mismatch_columns").reset_index(name="rows"), max_height=180)
    display(Markdown("#### Selected mismatch locator"))
    display_mismatch_detail(
        source_index=source_index,
        compare_mode=compare_mode,
        mismatch_index=mismatch_index,
        tolerance=tolerance,
        status=status,
        xml_id_filter=xml_id,
        context=max(2, min(8, rows // 2)),
    )
    display(Markdown("#### Diff rows"))
    display_scrollable_df(cmp.head(rows), max_height=360)
    display(Markdown("#### Raw df_pitch side by side"))
    display_side_by_side(part.iloc[row_start:row_start + rows], vrv.iloc[row_start:row_start + rows], n=rows)


if widgets is not None:
    source_dropdown = widgets.Dropdown(
        options=[(name, idx) for idx, name in enumerate(source_names)],
        value=0,
        description="Source",
        layout=widgets.Layout(width="650px"),
    )
    status_dropdown = widgets.Dropdown(
        options=["all", "value_mismatch", "missing_in_verovio", "extra_in_verovio", "match"],
        value="value_mismatch",
        description="Status",
    )
    compare_dropdown = widgets.Dropdown(
        options=list(COMPARE_COLUMN_PRESETS.keys()),
        value="core",
        description="Compare",
    )
    row_start = widgets.IntText(value=0, description="Row start")
    row_count = widgets.IntSlider(value=25, min=5, max=100, step=5, description="Rows")
    mismatch_index = widgets.IntText(value=0, description="Mismatch #")
    tolerance = widgets.FloatLogSlider(value=1e-6, base=10, min=-9, max=-1, step=1, description="Tol")
    xml_filter = widgets.Text(value="", description="xml_id", placeholder="optional substring")
    out = widgets.Output(layout=widgets.Layout(max_height="900px", overflow_y="auto", border="1px solid #ddd"))

    def _refresh(*_args):
        with out:
            render_comparison_view(
                source_dropdown.value,
                compare_dropdown.value,
                status_dropdown.value,
                mismatch_index.value,
                row_start.value,
                row_count.value,
                tolerance.value,
                xml_filter.value,
            )

    for widget in (source_dropdown, compare_dropdown, status_dropdown, mismatch_index, row_start, row_count, tolerance, xml_filter):
        widget.observe(_refresh, names="value")

    display(widgets.VBox([
        widgets.HBox([source_dropdown]),
        widgets.HBox([compare_dropdown, status_dropdown, mismatch_index, row_count, tolerance]),
        widgets.HBox([xml_filter, row_start]),
        out,
    ]))
    _refresh()
else:
    render_comparison_view(source_index=0, compare_mode="core", status="value_mismatch", mismatch_index=0, row_start=0, rows=25, tolerance=1e-6)


In [6]:
# Event coverage overview. This is useful when df_pitch is close but annotations differ.

def event_type_summary(source_index: int = 0) -> pd.DataFrame:
    rows = []
    for backend in BACKENDS:
        events = get_events_df(backend, source_index)
        counts = events.get("type", pd.Series(dtype=object)).value_counts(dropna=False)
        for event_type, count in counts.items():
            rows.append({"backend": backend, "type": event_type, "count": int(count)})
    if not rows:
        return pd.DataFrame(columns=["backend", "type", "count"])
    table = pd.DataFrame(rows)
    return table.pivot_table(index="type", columns="backend", values="count", fill_value=0, aggfunc="sum").reset_index()


display(event_type_summary(source_index=0))


backend,type,partitura,verovio
0,direction,2,2
1,fermata,10,10
2,measure,427,427
3,mrest,792,792
4,rest,3607,3607
5,slur,112,112
6,tempo,2,2
7,tie,492,492
8,trill,20,20


In [7]:
# Manual deep-dive cell. Change these values while debugging a concrete mismatch.

SOURCE_INDEX = 0
COMPARE_MODE = "core"  # core | timing | strict | location
STATUS_FILTER = "value_mismatch"  # all | match | value_mismatch | missing_in_verovio | extra_in_verovio
XML_ID_FILTER = ""
MISMATCH_INDEX = 0
ROW_START = 0
ROW_COUNT = 25
TOLERANCE = 1e-6

cmp = filtered_mismatches(
    source_index=SOURCE_INDEX,
    compare_mode=COMPARE_MODE,
    status=STATUS_FILTER,
    tolerance=TOLERANCE,
    xml_id_filter=XML_ID_FILTER,
)

display(parity_summary(SOURCE_INDEX, tolerance=TOLERANCE, compare_columns=COMPARE_COLUMN_PRESETS[COMPARE_MODE]))
display(cmp["mismatch_columns"].value_counts().rename_axis("mismatch_columns").reset_index(name="rows"))
display(cmp.head(ROW_COUNT))
display_mismatch_detail(
    source_index=SOURCE_INDEX,
    compare_mode=COMPARE_MODE,
    mismatch_index=MISMATCH_INDEX,
    tolerance=TOLERANCE,
    status=STATUS_FILTER,
    xml_id_filter=XML_ID_FILTER,
    context=5,
)
show_pitch_window(SOURCE_INDEX, start=ROW_START, n=ROW_COUNT)


,source,partitura_rows,verovio_rows,partitura_xml_ids,verovio_xml_ids,matches,value_mismatches,missing_in_verovio,extra_in_verovio
0,00_bach_js_brandenburgconcert_no4_i_bwv1049,10763,10763,10763,10763,10763,0,0,0


,mismatch_columns,rows


,xml_id,MIDI_partitura,Global Onset_partitura,Local Onset_partitura,Duration_partitura,Pitch_partitura,Pitch Enharmonic_partitura,Voice_partitura,Measure_partitura,MIDI_verovio,Global Onset_verovio,Local Onset_verovio,Duration_verovio,Pitch_verovio,Pitch Enharmonic_verovio,Voice_verovio,Measure_verovio,_merge,status,mismatch_columns


No rows match the current mismatch filters.

Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
1,0.00,0.00,0.50,G3,G3,55,Cello - Voice 1,d1e780
1,0.00,0.00,0.50,G3,G3,55,Double Bass - Voice 1,d1e839
1,0.00,0.00,0.50,G3,G3,55,w15rfroz - Voice 1,d1e974
1,0.00,0.00,0.50,B4,B4,71,Viola - Voice 1,d1e728
1,0.00,0.00,0.50,B4,B4,71,P9-I9 - Voice 1,d1e901
1,0.00,0.00,0.50,D5,D5,74,Violin II - Voice 1,d1e676
1,0.00,0.00,0.50,D5,D5,74,P9-I9 - Voice 1,d1e917
1,0.00,0.00,0.50,G5,G5,79,Solo Violin - Voice 1,d1e402
1,0.00,0.00,0.50,G5,G5,79,Violin I - Voice 1,d1e624
1,0.00,0.00,0.50,G5,G5,79,P9-I9 - Voice 1,d1e934
